# Collaborative Filtering for Polymarket

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from implicit.als import AlternatingLeastSquares
from collections import defaultdict
from dotenv import load_dotenv
from sqlalchemy import create_engine

c:\Users\euseb\anaconda3\envs\tools\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## load data

In [2]:
from utils import load_trade_tags

df = load_trade_tags()

print(f"trades: {len(df):,}")
print(f"users: {df['user_id'].nunique():,}")
print(f"tags: {df['tag_id'].nunique():,}")
df.head()

No cache, querying database...
Saving 14,111,477 rows to 82 files in 'df_trade_tags_cache/'
done
trades: 14,111,477
users: 8,581
tags: 3,721


,user_id,tag_id,tag_label,timestamp,trade_count
0,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,100215,All,2022-11-22 03:59:32,1
1,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,126,Trump,2022-11-22 03:59:32,1
2,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,143,u.s. 2024 republican presidential nomination,2022-11-22 03:59:32,1
3,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,144,Elections,2022-11-22 03:59:32,1
4,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,160,ron desantis,2022-11-22 03:59:32,1


## train/test split

In [3]:
# taking timestamp to split 80/20
df = df.sort_values(['user_id', 'timestamp'])

df['row_num'] = df.groupby('user_id').cumcount()
df['total'] = df.groupby('user_id')['timestamp'].transform('size')
df['cutoff'] = (df['total'] * 0.8).astype(int)
df['cutoff'] = df['cutoff'].clip(lower=1, upper=df['total']-1)
df = df[df['total'] >= 2]

# filtering users with at leat 2 trades
# This remove arund 5 users, because i encoutered some issues
train_df = df[df['row_num'] < df['cutoff']].copy()
test_df = df[df['row_num'] >= df['cutoff']].copy()

for col in ['row_num', 'total', 'cutoff']:
    train_df.drop(col, axis=1, inplace=True)
    test_df.drop(col, axis=1, inplace=True)

print(f"train: {len(train_df):,}")
print(f"test: {len(test_df):,}")

train: 11,285,771
test: 2,825,704


In [4]:
# using trade counts
train_agg = train_df.groupby(['user_id', 'tag_id', 'tag_label'])['trade_count'].sum().reset_index()
test_agg = test_df.groupby(['user_id', 'tag_id', 'tag_label'])['trade_count'].sum().reset_index()

print(f"train pairs: {len(train_agg):,}")
print(f"test pairs: {len(test_agg):,}")

train pairs: 585,126
test pairs: 269,099


## build matrix

In [5]:
# indexes for the matrix 
users_cat = train_agg['user_id'].astype('category')
tags_cat = train_agg['tag_id'].astype('category')

user_to_id = {u: i for i, u in enumerate(users_cat.cat.categories)}
id_to_user = {i: u for u, i in user_to_id.items()}
tag_to_id = {t: i for i, t in enumerate(tags_cat.cat.categories)}
id_to_tag = {i: t for t, i in tag_to_id.items()}

n_users = len(user_to_id)
n_tags = len(tag_to_id)
print(f"matrix: {n_users} x {n_tags}")

matrix: 8579 x 3666


In [6]:
alpha = 40

row_id = users_cat.cat.codes.values
col_id = tags_cat.cat.codes.values
# confidence = 1 + alpha * log(1 + trades)
confidence = 1 + alpha * np.log1p(train_agg['trade_count'].values)

matrix = sparse.csr_matrix((confidence, (row_id, col_id)), shape=(n_users, n_tags))

density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
print(f"density: {density:.4f} ({density*100:.2f}%)")

density: 0.0186 (1.86%)


## train

In [13]:
model = AlternatingLeastSquares(
    factors=64,     # factors 64 seems to have best results 
    regularization=0.05, #  prevent overfitting
    iterations=30,
    random_state=42
)

model.fit(matrix, show_progress=True)
model.save("../models/collaborative_filtering_model.npz")

100%|██████████| 30/30 [00:02<00:00, 11.11it/s]


## evaluate

In [8]:
test_tags_by_user = test_agg.groupby('user_id')['tag_id'].apply(set).to_dict()

k = 10
precisions = []
hits = 0
total = 0

for user_id, true_tags in test_tags_by_user.items():
    if user_id not in user_to_id:
        continue
    
    u_id = user_to_id[user_id]
    rec_ids, scores = model.recommend(u_id, matrix[u_id], N=k, filter_already_liked_items=True)
    rec_tags = {id_to_tag[i] for i in rec_ids if i in id_to_tag}
    
    n_hits = len(rec_tags & true_tags)
    precisions.append(n_hits / k)
    
    if n_hits > 0:
        hits += 1
    total += 1

print(f"precision@{k}: {np.mean(precisions):.4f}")
print(f"hit rate@{k}: {hits/total:.4f}")
print(f"users: {total}")

precision@10: 0.1006
hit rate@10: 0.4985
users: 8579


## recommend events

In [9]:
from utils import load_events_with_tags

events_df = load_events_with_tags()
events_df['event_id'].nunique()

No events cache, querying database...
Saving 744,626 rows to 7 files in 'events_tag_cache/'
done


65056

In [10]:
event_tags = defaultdict(list)
event_titles = {}
tag_labels = {}

for _, row in events_df.iterrows():
    event_id = row['event_id']
    tag_id = row['tag_id']
    
    if tag_id in tag_to_id:
        t_id = tag_to_id[tag_id]
        event_tags[event_id].append(t_id)
        tag_labels[t_id] = row['tag_label']
    
    event_titles[event_id] = row['title']
len(event_tags)

65056

In [11]:
def get_event_recs(user_id, n=5):
    if user_id not in user_to_id:
        return None
    
    # get users tag preferences
    u_id = user_to_id[user_id]
    user_vec = model.user_factors[u_id]
    tag_scores = model.item_factors.dot(user_vec)
    
     #score events by averaging tag scores
    scores = []
    for event_id, t_ids in event_tags.items():
        avg_score = np.mean([tag_scores[t] for t in t_ids])
        scores.append((event_id, avg_score))
    
    scores.sort(key=lambda x: -x[1])
    
    results = []
    for event_id, score in scores[:n]:
        tags = [tag_labels[t] for t in event_tags[event_id]]
        results.append({
            'title': event_titles[event_id],
            'score': score,
            'tags': set(tags)
        })
    
    return results

In [12]:
# test
sample_user = "0x3428d8d085e8b9900768dcded987fba284694500"#list(user_to_id.keys())[10]
recs = get_event_recs(sample_user, n=5)

print(f"user: {sample_user}")
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['title']}")
    print(f"\ttags: {r['tags']}")
    print(f"\tscore: {r['score']:.3f}\n")

user: 0x3428d8d085e8b9900768dcded987fba284694500
1. Solana flips ETH in daily fees in July?
	tags: {'Ethereum', 'Solana', 'Crypto'}
	score: 1.073

2. Will ETH hard fork before May?
	tags: {'Ethereum', 'Crypto'}
	score: 1.071

3. Ethereum ETF begins trading by July 26?
	tags: {'Ethereum', 'Crypto'}
	score: 1.071

4. Solana flips ETH in daily fees in June?
	tags: {'Ethereum', 'Crypto Prices', 'Solana', 'Crypto'}
	score: 1.067

5. SOLETH hit 0.08 by Nov 30?
	tags: {'Ethereum', 'Crypto Prices', 'Solana', 'Crypto'}
	score: 1.067

